**Data Cleanup**

In [1]:
"""
Data we need
- set of exams to schedule
- list of timeslots
- list of rooms available, by timeslot
- set of unique student schedules
- number of students with each unique schedule
- where classes during the semester
- set of classes that need to be split across rooms

Hard Constraints:
- room capacities
- all exams scheduled


Factors to consider:
1. student overlaps
2. 3 exams in 48 hours
3. back-to-back exams
4. in-person exams finish as early as possible
5. accomodate room requests
6. correct room type

If we have time:
- minimize exams on sunday
- fairness across majors

"""

'\nData we need\n- set of exams to schedule\n- list of timeslots\n- list of rooms available, by timeslot\n- set of unique student schedules\n- number of students with each unique schedule\n- where classes during the semester\n- set of classes that need to be split across rooms\n\nHard Constraints:\n- room capacities\n- all exams scheduled\n\n\nFactors to consider:\n1. student overlaps\n2. 3 exams in 48 hours\n3. back-to-back exams\n4. in-person exams finish as early as possible\n5. accomodate room requests\n6. correct room type\n\nIf we have time:\n- minimize exams on sunday\n- fairness across majors\n\n'

In [ ]:
import pandas as pd

# Load the CSV files
class_info = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Class_Info2023.csv')
final_exams = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Final_Exams2023.csv')
schedule = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/Schedule2023.csv')
student_reg = pd.read_csv('/Users/benfreidinger/Desktop/decisionlab26/past_group_work/2023-2024/StudentRegistration2023.csv')

# Process exams (E): list of CRNs from final_exams
E = final_exams['CRN'].tolist()

# Class sizes: number of students per exam (CRN)
class_sizes = student_reg.groupby('CRN').size().to_dict()

# Timeslots (T): unique (EXAM_DATE, EXAM_TIME) from final_exams
timeslots = final_exams[['EXAM_DATE', 'EXAM_TIME']].drop_duplicates()
T = [(row['EXAM_DATE'], row['EXAM_TIME']) for _, row in timeslots.iterrows()]

# Rooms (R): unique rooms from class_info where AVAILABLE_TO_SCHEDULE == 'Y'
available_rooms = class_info[class_info['AVAILABLE_TO_SCHEDULE'] == 'Y']
rooms = available_rooms[['BUILDING_CODE', 'ROOM_NUMBER']].drop_duplicates()
R = [(row['BUILDING_CODE'], row['ROOM_NUMBER']) for _, row in rooms.iterrows()]


# Capacities: dict room -> ROOM_CAPACITY
capacities = {(row['BUILDING_CODE'], row['ROOM_NUMBER']): row['ROOM_CAPACITY'] for _, row in available_rooms.iterrows()}
capacities[('dummy', '1')] = capacities[('HRZ','AMP')] + capacities[('KCK', '100')]
capacities[('dummy', '2')] = capacities[('DCH', '1055')] + capacities[('HRG', '100')]
capacities[('dummy', '3')] = capacities[('HRZ', '210')] + capacities[('HRZ', '212')]

# Student schedules (S): dict index -> set of CRNs
student_schedules = student_reg.groupby('PERSON_IDENTIFIER')['CRN'].apply(set).to_dict()

# Unique schedules and counts (S and N)
from collections import defaultdict
schedule_counts = defaultdict(int)
schedule_to_index = {}
index = 0
for sched in student_schedules.values():
    sched_tuple = frozenset(sched)
    if sched_tuple not in schedule_to_index:
        schedule_to_index[sched_tuple] = index
        index += 1
    schedule_counts[schedule_to_index[sched_tuple]] += 1

S = {idx: set(sched) for sched, idx in schedule_to_index.items()}
N = schedule_counts

print(f"E = {E}")
print(f"T = {T}")

E = [15400, 13127, 15395, 15397, 13495, 15434, 15435, 14096, 14353, 14765, 15432, 15496, 15505, 10516, 11988, 12199, 12244, 12540, 12541, 14110, 15820, 14776, 10209, 15827, 15379, 13380, 14358, 14362, 14401, 15648, 15256, 15257, 15258, 15486, 15488, 16183, 16325, 14787, 15821, 15547, 15422, 15423, 15426, 10305, 11113, 16108, 12100, 15673, 15571, 15572, 11689, 16365, 16366, 10282, 12810, 12306, 12740, 14056, 10084, 10087, 16386, 16387, 16399, 16465, 11690, 11879, 15644, 16284, 15529, 15611, 15800, 15581, 15582, 15583, 15584, 16367, 16368, 16374, 16375, 16471, 11738, 11752, 11753, 11775, 12032, 13633, 13740, 16424, 12147, 12851, 13913, 13916, 16106, 16332, 14334, 10086, 14345, 14391, 14671, 16285, 15807, 15535, 15536, 16017, 16083, 16383, 10300, 10299, 11671, 14903, 10594, 11585, 11586, 12557, 12636, 13499, 14347, 15194, 15779, 16070, 16071, 10416, 15525, 14701, 13327, 13647, 13648, 13650, 13651, 15585, 15586, 15587, 15588, 15602, 15603, 15604, 15605, 15607, 15608, 15609, 15610, 16068, 1

In [ ]:
import gurobipy as gp
from gurobipy import GRB

env = gp.Env(empty=True)
env.setParam("OutputFlag", 0)
env.start()

m = gp.Model(env=env)

def schedule_model(E, T, R, S, N, capacities, class_sizes):
    """
    Inputs:
    E - list of CRNs (exams to schedule)
    T - ordered list of (EXAM_DATE, EXAM_TIME) timeslot tuples
    R - list of (BUILDING_CODE, ROOM_NUMBER) room tuples
    S - dict: schedule_idx -> set of CRNs
    N - dict: schedule_idx -> number of students with that schedule
    capacities - dict: (BUILDING_CODE, ROOM_NUMBER) -> capacity
    class_sizes - dict: CRN -> number of enrolled students
    """
    E_set = set(E)
    sched_keys = list(S.keys())

    # CRNs per schedule that have a final exam
    exam_crns_by_sched = {
        idx: [crn for crn in crns if crn in E_set]
        for idx, crns in S.items()
    }

    # Ordered CRN pairs for overlap/z variables
    E_pairs = [(E[i], E[j]) for i in range(len(E)) for j in range(i + 1, len(E))]

    # DECISION VARIABLES indexed by set elements
    x = m.addVars(E, T, R, vtype=GRB.BINARY, name="x")
    y = m.addVars(E, T, vtype=GRB.BINARY, name="y")
    z = m.addVars(E_pairs, T, vtype=GRB.CONTINUOUS, name="z")
    
    # variables for dummy rooms (combined two rooms)
    dummy1 = [('dummy', '1')]
    dummy2 = [('dummy', '2')]
    dummy3 = [('dummy', '3')]
    d1 = m.addVars(E, T, dummy1, vtype=GRB.BINARY, name='d1')
    d2 = m.addVars(E, T, dummy2, vtype=GRB.BINARY, name='d2')
    d3 = m.addVars(E, T, dummy3, vtype=GRB.BINARY, name='d3')

    m.addConstrs(2 * (1 - d1[e, t, ('dummy', '1')]) >= x[e,t,('HRZ','AMP')] + x[e,t,('KCK', '100')] for e in E for t in T)
    m.addConstrs(2 * (1 - d2[e, t, ('dummy', '2')]) >= x[e,t,('DCH','1055')] + x[e,t,('HRG', '100')] for e in E for t in T)
    m.addConstrs(2 * (1 - d3[e, t, ('dummy', '3')]) >= x[e,t,('HRZ','210')] + x[e,t,('HRZ', '212')] for e in E for t in T)

    # Room capacities for dummy rooms
    m.addConstrs(
        (gp.quicksum(class_sizes[crn] * d1[crn, t, ('dummy', '1')] for crn in E) <= capacities[('dummy', '1')]
         for t in T for r in R),
        name="capacity_dummy1")
    m.addConstrs(
        (gp.quicksum(class_sizes[crn] * d2[crn, t, ('dummy', '2')] for crn in E) <= capacities[('dummy', '2')]
         for t in T for r in R),
        name="capacity_dummy2")
    m.addConstrs(
        (gp.quicksum(class_sizes[crn] * d3[crn, t, ('dummy', '3')] for crn in E) <= capacities[('dummy', '3')]
         for t in T for r in R),
        name="capacity_dummy3")



    w = {t: 1 for t in T}  # timeslot weights (late-penalty placeholder)

    # Student overlap counts keyed by CRN pair
    overlap = {}
    for crn1, crn2 in E_pairs:
        overlap[(crn1, crn2)] = sum(
            N[idx] for idx, crns in S.items()
            if crn1 in crns and crn2 in crns
        )

    # TRACKING 3 EXAMS IN 48 HOURS
    # v[t_start, sched_idx] = # exams for sched_idx in the 6-slot window starting at t_start
    window_starts = T[:-5]
    v = m.addVars(window_starts, sched_keys, vtype=GRB.INTEGER, name="v")

    for t_idx, t_start in enumerate(window_starts):
        window = T[t_idx:t_idx + 6]
        for sched_idx in sched_keys:
            m.addConstr(
                v[t_start, sched_idx] == gp.quicksum(
                    x[crn, t, r]
                    for crn in exam_crns_by_sched[sched_idx]
                    for t in window
                    for r in R
                ),
                name=f"v_def[{t_start},{sched_idx}]"
            )

    u = m.addVars(window_starts, sched_keys, vtype=GRB.BINARY, name="u")
    for t_start in window_starts:
        for sched_idx in sched_keys:
            m.addGenConstrIndicator(u[t_start, sched_idx], True,  v[t_start, sched_idx] >= 3)
            m.addGenConstrIndicator(u[t_start, sched_idx], False, v[t_start, sched_idx] <= 2)

    # TRACKING BACK-TO-BACK EXAMS
    consec_starts = T[:-1]
    d = m.addVars(consec_starts, sched_keys, vtype=GRB.INTEGER, name="d")

    for t_idx, t in enumerate(consec_starts):
        t_next = T[t_idx + 1]
        for sched_idx in sched_keys:
            crns = exam_crns_by_sched[sched_idx]
            m.addConstr(d[t, sched_idx] <= gp.quicksum(x[crn, t,      r] for crn in crns for r in R))
            m.addConstr(d[t, sched_idx] <= gp.quicksum(x[crn, t_next, r] for crn in crns for r in R))
            m.addConstr(1 + d[t, sched_idx] >= gp.quicksum(
                x[crn, t, r] + x[crn, t_next, r] for crn in crns for r in R
            ))

    # OBJECTIVE
    lamb = 1
    mu   = 10  # penalty for 3 exams in 48 hours
    nu   = 5   # penalty for back-to-back

    m.setObjective(
        gp.quicksum(w[t] * x[crn, t, r] for crn in E for t in T for r in R) +
        lamb * gp.quicksum(overlap[(c1, c2)] * z[c1, c2, t] for c1, c2 in E_pairs for t in T) +
        mu   * gp.quicksum(u[t_start, idx]  for t_start in window_starts for idx in sched_keys) +
        nu   * gp.quicksum(d[t, idx]        for t in consec_starts        for idx in sched_keys),
        GRB.MINIMIZE
    )

    # HARD CONSTRAINTS

    # Each exam scheduled exactly once
    m.addConstrs(
        (gp.quicksum(x[crn, t, r] for t in T for r in R) == 1 for crn in E),
        name="assign_once"
    )

    # Room capacities
    m.addConstrs(
        (gp.quicksum(class_sizes[crn] * x[crn, t, r] for crn in E) <= capacities[r]
         for t in T for r in R),
        name="capacity"
    )

    # y[crn, t] = 1 iff crn is scheduled at t
    m.addConstrs(
        (y[crn, t] == gp.quicksum(x[crn, t, r] for r in R) for crn in E for t in T),
        name="y_def"
    )

    # Linearize z[c1, c2, t] = y[c1, t] * y[c2, t]
    m.addConstrs((z[c1, c2, t] <= y[c1, t]                 for c1, c2 in E_pairs for t in T))
    m.addConstrs((z[c1, c2, t] <= y[c2, t]                 for c1, c2 in E_pairs for t in T))
    m.addConstrs((z[c1, c2, t] >= y[c1, t] + y[c2, t] - 1 for c1, c2 in E_pairs for t in T))

    m.optimize()

    print("Optimal objective: {}".format(m.ObjVal))

In [ ]:
schedule_model(E,T,R,S,N, capacities, class_sizes)